# Introduction to Deep Learning

Deep learning is a branch of machine learning that uses artificial neural networks with multiple layers to learn patterns from data. These networks can automatically build useful representations, progressing from simple features to more complex ones.

Deep learning is used in areas such as image recognition, natural language processing, speech recognition, and recommendation systems. It can perform remarkably well when trained on large datasets, though it often requires substantial computing resources and careful evaluation.

In [ ]:
import re
import tensorflow as tf
from tensorflow.keras import layers, models

vocab_size = 20_000
max_length = 250

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(
    num_words=vocab_size
)

x_train = tf.keras.utils.pad_sequences(
    x_train, maxlen=max_length, padding="post", truncating="post"
)
x_test = tf.keras.utils.pad_sequences(
    x_test, maxlen=max_length, padding="post", truncating="post"
)

model = models.Sequential([
    layers.Input(shape=(max_length,)),
    layers.Embedding(vocab_size, 64, mask_zero=True),
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, validation_split=0.2, epochs=3, batch_size=64)

test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_accuracy:.3f}")

word_index = tf.keras.datasets.imdb.get_word_index()

def predict_sentiment(text):
    tokens = re.findall(r"\b\w+\b", text.lower())
    sequence = [
        word_index[word] + 3
        for word in tokens
        if word in word_index and word_index[word] + 3 < vocab_size
    ]
    sequence = tf.keras.utils.pad_sequences(
        [sequence], maxlen=max_length, padding="post", truncating="post"
    )
    score = float(model.predict(sequence, verbose=0)[0][0])
    return {"sentiment": "positive" if score >= 0.5 else "negative", "score": score}

predict_sentiment("The story was engaging and the performances were excellent.")

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
time = np.arange(2500)
series = (
    0.002 * time
    + np.sin(2 * np.pi * time / 50)
    + 0.5 * np.sin(2 * np.pi * time / 200)
    + rng.normal(0, 0.1, size=len(time))
).astype(np.float32)

window_size = 48
windows = np.array([
    series[i : i + window_size]
    for i in range(len(series) - window_size)
])
targets = series[window_size:]

split = int(len(windows) * 0.8)
x_train_ts, x_test_ts = windows[:split], windows[split:]
y_train_ts, y_test_ts = targets[:split], targets[split:]

time_series_model = models.Sequential([
    layers.Input(shape=(window_size, 1)),
    layers.LSTM(32),
    layers.Dense(1),
])

time_series_model.compile(optimizer="adam", loss="mse", metrics=["mae"])
time_series_model.fit(
    x_train_ts[..., np.newaxis],
    y_train_ts,
    validation_split=0.2,
    epochs=5,
    batch_size=32,
    verbose=1,
)

test_loss, test_mae = time_series_model.evaluate(
    x_test_ts[..., np.newaxis], y_test_ts, verbose=0
)
print(f"Test MAE: {test_mae:.3f}")

next_value = time_series_model.predict(
    series[-window_size:][np.newaxis, :, np.newaxis], verbose=0
)[0, 0]
print(f"Predicted next value: {next_value:.3f}")

In [ ]:
(x_train_digits, y_train_digits), (x_test_digits, y_test_digits) = (
    tf.keras.datasets.mnist.load_data()
)

x_train_digits = x_train_digits.astype("float32")[..., tf.newaxis] / 255.0
x_test_digits = x_test_digits.astype("float32")[..., tf.newaxis] / 255.0

digit_model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Conv2D(64, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
])

digit_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

digit_model.fit(
    x_train_digits,
    y_train_digits,
    epochs=3,
    batch_size=64,
    validation_split=0.1,
)

test_loss, test_accuracy = digit_model.evaluate(x_test_digits, y_test_digits, verbose=0)
print(f"Test accuracy: {test_accuracy:.3f}")

prediction = digit_model.predict(x_test_digits[:1], verbose=0)[0]
print(f"Predicted digit: {prediction.argmax()}, actual digit: {y_test_digits[0]}")